In [1]:
import os
import sys
sys.path.insert(0, '')
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import torch
import numpy as np
from PIL import Image
from model import AdaptModel, data_transform, inverse_data_transform
from torchvision import transforms
from matplotlib import pyplot as plt
from tqdm import tqdm
from thop import profile

rank = 'cuda:0'
model = AdaptModel(rank=rank, num_class=9, pretrained='./pretrained/MKF/model.pth')
model.ae.encoder = model.ae.encoder.to(rank)
model.ae.decoder = model.ae.decoder.to(rank)
model.IKRModule = model.IKRModule.to(rank)
model.CKGModule = model.CKGModule.to(rank)
model.FusionModule = model.FusionModule.to(rank)
model.FusionModule.eval()
print('done')

[Info]: Loading config from [./pretrained/IKR/config.yaml] as yaml file.
[Info]: Loading config from [./pretrained/CKG/config.yaml] as yaml file.
==> load pretrained model
done


In [2]:
skip = model.num_timesteps // 25
seq = range(0, model.num_timesteps, skip)
seq_next = [-1] + list(seq[:-1])

In [3]:
color_map = {
    0: [0, 0, 0],       # unlabelled
    1: [64, 0, 128],    # car
    2: [64, 64, 0],     # person
    3: [0, 128, 192],   # bike
    4: [0, 0, 192],     # curve
    5: [128, 128, 0],   # car_stop
    6: [64, 64, 128],   # guardrail
    7: [192, 128, 128], # color_cone
    8: [192, 64, 0],     # bump
}
def visualize_segmentation(label_map):
    h, w = label_map.shape
    colored = np.zeros((h, w, 3), dtype=np.uint8)
    for class_id, color in color_map.items():
        colored[label_map == class_id] = color
    return colored

In [18]:
datapath = './data/vis'
filename = '00122.png'

In [ ]:
seed = 55
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

image = Image.open(os.path.join(datapath, filename)).convert('RGB')
print(image.size)
img = transforms.ToTensor()(image).unsqueeze(0).to(rank)

raw_z = model.ae.encoder(img)
raw_z = data_transform(raw_z) 

ikr_xt = ckg_xt = fuse_xt = torch.randn_like(raw_z, dtype=torch.float32).to(rank)  
batch, c, h, w = raw_z.shape
seg_xt = torch.randn(batch, 9, h, w).to(rank)

with torch.no_grad():
    for it, (i, j) in tqdm(enumerate(zip(reversed(seq), reversed(seq_next)))):
        t = (torch.ones(batch) * i).to(rank)
        next_t = (torch.ones(batch) * j).to(rank)
        at = model.alphas_cumprod_pre.index_select(0, t.long()+1).view(-1, 1, 1, 1)
        at_next = model.alphas_cumprod_pre.index_select(0, next_t.long()+1).view(-1, 1, 1, 1)
        ikr_noise, ikr_xt, ikr_x0 = model.ikr_sample_onestep(raw_z.float(), ikr_xt.float(), t, next_t)
        ckg_noise, ckg_xt, ckg_x0 = model.ckg_sample_onestep(raw_z.float(), ckg_xt.float(), t, next_t)
        noise, seg_map, seg_xt = model.FusionModule(torch.cat([ikr_x0, ckg_x0, fuse_xt], dim=1), ikr_noise, ckg_noise, t, seg_xt) 
        x0 = (fuse_xt - noise * (1 - at).sqrt()) / at.sqrt()
        c2 = (1 - at_next).sqrt()
        fuse_xt = at_next.sqrt() * x0 + c2 * noise
ikr_z = inverse_data_transform(ikr_x0)
ckg_z = inverse_data_transform(ckg_x0)
mkf_z = inverse_data_transform(x0)

ikrimg = model.ae.decoder(ikr_z)
ckgimg = model.ae.decoder(ckg_z)
MagImg = model.ae.decoder(mkf_z)

ikrimg=ikrimg.squeeze(0).permute(1, 2, 0)
ikrimg=ikrimg.cpu().numpy()

ckgimg=ckgimg.squeeze(0).permute(1, 2, 0)
ckgimg=ckgimg.cpu().numpy()

MagImg=MagImg.squeeze(0).permute(1, 2, 0)
MagImg=MagImg.cpu().numpy()

seg_map = torch.softmax(seg_map, dim=1)
seg_label = torch.argmax(seg_map, dim=1)
segimg=seg_label.squeeze(0).cpu().numpy()
segimg=visualize_segmentation(segimg)

# show result
_,ax = plt.subplots(2,3,figsize=(8,6))
plt.rcParams['axes.titlesize']=10
plt.subplots_adjust(hspace=0.2, wspace=0)
[a.axis('off') for a in ax.flatten()]
ax[0,0].imshow(image)
ax[0,0].set_title('raw')
ax[0,1].imshow(ikrimg)
ax[0,1].set_title('denoise')
ax[0,2].imshow(ckgimg)
ax[0,2].set_title('style')
ax[1,0].imshow(MagImg)
ax[1,0].set_title('fused')
ax[1,1].imshow(segimg)
ax[1,1].set_title('seg')
plt.show()

# save result
save_path = os.path.join('results')
os.makedirs(save_path, exist_ok=True)

MagImg = np.clip(MagImg * 255, 0, 255).astype(np.uint8)
MagImg = Image.fromarray(MagImg, mode='RGB')
MagImg.save(os.path.join(save_path, 'MagImg_'+filename)) 

segimg = Image.fromarray(segimg, mode='RGB')
segimg.save(os.path.join(save_path, 'seg_'+filename)) 

ikrimg = np.clip(ikrimg * 255, 0, 255).astype(np.uint8) 
ikrimg = Image.fromarray(ikrimg, mode='RGB')
ikrimg.save(os.path.join(save_path, 'ikr_'+filename)) 

ckgimg = np.clip(ckgimg * 255, 0, 255).astype(np.uint8) 
ckgimg = Image.fromarray(ckgimg, mode='RGB')
ckgimg.save(os.path.join(save_path, 'ckg_'+filename)) 
print(f'image saved to {save_path}')